In [ ]:
import datetime as dt
from dateutil.relativedelta import relativedelta
import data_provider.DataReader as data_provider
import config as cfg
import yahoo_fin.stock_info as yf
import pandas as pd
import backtrader as bt
import yfinance as yf


In [14]:
end_date = dt.datetime.now()
start_date = end_date - relativedelta(days=14)
data = data_provider.yFinanceReader().historic_price_data("xrp-usd", start_date, end_date)

In [15]:
data

,Open,High,Low,Close,Volume
Datetime,,,,,
2023-10-27 18:00:00+00:00,0.539995,0.544128,0.539981,0.544128,10817664
2023-10-27 19:30:00+00:00,0.544156,0.546602,0.544156,0.545823,7101696
2023-10-27 21:00:00+00:00,0.545846,0.546705,0.545677,0.546686,943808
2023-10-27 22:30:00+00:00,0.546699,0.546734,0.545704,0.546123,2192064
2023-10-28 00:00:00+00:00,0.545715,0.545715,0.541656,0.541711,10303424
...,...,...,...,...,...
2023-11-10 10:30:00+00:00,0.657339,0.660838,0.655548,0.660838,21144320
2023-11-10 12:00:00+00:00,0.660733,0.660733,0.652578,0.654461,17561600
2023-11-10 13:30:00+00:00,0.654562,0.655076,0.643111,0.644630,34463488


In [32]:
class MyStrategy(bt.Strategy):
    def __init__(self):
        self.rsi = bt.indicators.RSI_SMA(self.data.close, period=14)
        self.dataclose = self.datas[0].close

    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))


    def next(self):
        if not self.position:
            if self.rsi < 30:
                self.log('BUY CREATE, %.2f' % self.dataclose[0])
                self.log(self.rsi)
                self.buy(size=100)
        else:
            if self.rsi > 70:
                self.log('SELL CREATE, %.2f' % self.dataclose[0])

                self.sell(size=100)

In [33]:
cerebro = bt.Cerebro()
cerebro.addstrategy(MyStrategy)
cerebro.adddata(bt.feeds.PandasData(dataname=data))
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
cerebro.run()
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
cerebro.plot(iplot=False)

Starting Portfolio Value: 10000.00
2023-11-07, BUY CREATE, 0.68
2023-11-07, <backtrader.indicators.rsi.RSI_SMA object at 0x0000028679DE3CA0>
Final Portfolio Value: 9997.21


[[<Figure size 640x480 with 5 Axes>]]